# AeroNet Lite – Module 5A: Demand Forecasting
**BS Data Science – AI Semester Project SP2026**

This notebook trains regression models on the **Bike Sharing Dataset** to predict delivery demand.

Hourly bike rental count is used as a proxy for delivery demand — both reflect population behavior driven by time of day, weather, and weekday patterns.

---
**Models used:** Linear Regression, Random Forest Regressor  
**Metrics:** MAE, RMSE  
**Output:** Predicted demand values used to scale grid demand in the simulation

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

print('All imports successful.')

In [ ]:
# ── Cell 2: Load Dataset ─────────────────────────────────────────────────────
# Option A: Load real Bike Sharing CSV downloaded from Kaggle
# Place train.csv in aeronet_lite/data/raw/
DATA_PATH = '../data/raw/train.csv'

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    # Extract hour from datetime if present
    if 'datetime' in df.columns:
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['hour']  = df['datetime'].dt.hour
        df['day']   = df['datetime'].dt.dayofweek
        df['month'] = df['datetime'].dt.month
    print(f'Real dataset loaded: {df.shape}')
else:
    # Option B: Generate synthetic data with same structure
    print('train.csv not found. Generating synthetic Bike Sharing data...')
    np.random.seed(42)
    n = 10886
    hours     = np.tile(np.arange(24), n // 24 + 1)[:n]
    seasons   = np.random.randint(1, 5, n)
    holidays  = np.random.randint(0, 2, n)
    workdays  = np.random.randint(0, 2, n)
    weather   = np.random.randint(1, 5, n)
    temp      = np.random.uniform(0.1, 0.9, n)
    atemp     = (temp + np.random.uniform(-0.05, 0.05, n)).clip(0, 1)
    humidity  = np.random.uniform(0.2, 0.9, n)
    windspeed = np.random.uniform(0.0, 0.5, n)
    base  = 100 + 200 * temp + 50 * workdays
    peak  = 150 * np.sin(np.pi * hours / 24)
    count = (base + peak + np.random.normal(0, 30, n)).clip(1).astype(int)
    df = pd.DataFrame({
        'season': seasons, 'holiday': holidays, 'workingday': workdays,
        'weather': weather, 'temp': temp.round(4), 'atemp': atemp.round(4),
        'humidity': humidity.round(4), 'windspeed': windspeed.round(4),
        'hour': hours, 'count': count
    })
    print(f'Synthetic dataset created: {df.shape}')

df.head()

In [ ]:
# ── Cell 3: Exploratory Data Analysis ────────────────────────────────────────
print('Dataset Info:')
print(df.info())
print('\nDescriptive Statistics:')
df.describe()

In [ ]:
# ── Cell 4: Visualize Demand Distribution ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['count'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Demand Distribution')
axes[0].set_xlabel('Count (demand)')

if 'hour' in df.columns:
    hourly_avg = df.groupby('hour')['count'].mean()
    axes[1].plot(hourly_avg.index, hourly_avg.values, marker='o', color='darkorange')
    axes[1].set_title('Average Demand by Hour of Day')
    axes[1].set_xlabel('Hour')
    axes[1].set_ylabel('Avg Count')

if 'temp' in df.columns:
    axes[2].scatter(df['temp'], df['count'], alpha=0.1, color='green', s=2)
    axes[2].set_title('Temperature vs Demand')
    axes[2].set_xlabel('Temp (normalized)')
    axes[2].set_ylabel('Count')

plt.suptitle('Bike Sharing Demand – EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../report/figures/demand_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 5: Feature Engineering ──────────────────────────────────────────────
feature_cols = ['season', 'holiday', 'workingday', 'weather',
                'temp', 'atemp', 'humidity', 'windspeed']

if 'hour' in df.columns:
    feature_cols.append('hour')
if 'day' in df.columns:
    feature_cols.append('day')
if 'month' in df.columns:
    feature_cols.append('month')

# Keep only columns that exist
feature_cols = [f for f in feature_cols if f in df.columns]

X = df[feature_cols].copy()
y = df['count'].copy()

# Drop rows with NaN
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

print(f'Features used: {feature_cols}')
print(f'X shape: {X.shape}, y shape: {y.shape}')

In [ ]:
# ── Cell 6: Train/Test Split ──────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train size: {X_train.shape[0]}')
print(f'Test size : {X_test.shape[0]}')

In [ ]:
# ── Cell 7: Linear Regression ────────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)

print('Linear Regression Results')
print(f'  MAE  : {mae_lr:.2f}')
print(f'  RMSE : {rmse_lr:.2f}')
print(f'  R²   : {r2_lr:.4f}')

In [ ]:
# ── Cell 8: Random Forest Regressor ──────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf   = r2_score(y_test, y_pred_rf)

print('Random Forest Regressor Results')
print(f'  MAE  : {mae_rf:.2f}')
print(f'  RMSE : {rmse_rf:.2f}')
print(f'  R²   : {r2_rf:.4f}')

In [ ]:
# ── Cell 9: Model Comparison Table ───────────────────────────────────────────
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE':   [round(mae_lr, 2),  round(mae_rf, 2)],
    'RMSE':  [round(rmse_lr, 2), round(rmse_rf, 2)],
    'R²':    [round(r2_lr, 4),   round(r2_rf, 4)]
})
print(comparison.to_string(index=False))

In [ ]:
# ── Cell 10: Actual vs Predicted Plot ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in zip(axes,
    [y_pred_lr, y_pred_rf],
    ['Linear Regression', 'Random Forest']):

    ax.scatter(y_test[:200], y_pred[:200], alpha=0.4, s=10, color='steelblue')
    ax.plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()], 'r--', lw=1.5, label='Perfect Fit')
    ax.set_title(f'{title} – Actual vs Predicted')
    ax.set_xlabel('Actual Demand')
    ax.set_ylabel('Predicted Demand')
    ax.legend()

plt.suptitle('Demand Forecasting – Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../report/figures/demand_actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 11: Feature Importance (Random Forest) ───────────────────────────────
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
importances.plot(kind='bar', color='teal', edgecolor='white')
plt.title('Random Forest – Feature Importance', fontsize=12, fontweight='bold')
plt.ylabel('Importance')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../report/figures/demand_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(importances)

In [ ]:
# ── Cell 12: Grid Demand Scaling ──────────────────────────────────────────────
# Use predicted avg demand to scale the 10x10 simulation grid
avg_predicted = float(y_pred_rf.mean())
max_predicted = float(y_pred_rf.max())

print(f'Average predicted demand : {avg_predicted:.2f}')
print(f'Max predicted demand     : {max_predicted:.2f}')
print()
print('These values will be normalized and mapped to grid cells in the simulation.')
print('Residential cells get higher demand, Industrial/Open Field get lower demand.')

# Example: map predicted demand to 0-10 scale for grid
normalized_avg = round(10 * avg_predicted / max_predicted, 2)
print(f'Normalized grid demand (0-10 scale): {normalized_avg}')

In [ ]:
# ── Cell 13: Save Processed Demand Data ───────────────────────────────────────
os.makedirs('../data/processed', exist_ok=True)

# Save test predictions for reference
pred_df = X_test.copy()
pred_df['actual_demand']    = y_test.values
pred_df['predicted_demand'] = y_pred_rf
pred_df.to_csv('../data/processed/demand_predictions.csv', index=False)
print('Saved: data/processed/demand_predictions.csv')

print('\n=== Demand Forecasting Complete ===')
print(f'Best Model : Random Forest')
print(f'MAE        : {mae_rf:.2f}')
print(f'RMSE       : {rmse_rf:.2f}')
print(f'R²         : {r2_rf:.4f}')